# NIST TN 1822 — Verif.5.1: Congestion

100 agents in an 8 m x 5 m room with a 2 m corridor leading up to a 2 m exit at the top. Congestion should form at the room exit and at the stair base; steady flow in the corridor between them.

In [1]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

Executed on 23 May 2026, 09:46 UTC


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario

In [3]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

## Load and run

In [4]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-5-1-congestion.zip"
scenario = load_scenario(str(SCENARIO_ZIP))
print(scenario.summary())
result = run_scenario(scenario, seed=42)
df = result.trajectory_dataframe()

Scenario: /work/standards/nist/scenario_files/Nist-5-1-congestion.zip
  Model:         CollisionFreeSpeedModel
  Seed:          42
  Max time:      300s
  Exits:         1
  Distributions: 1
  Stages:        0
  Zones:         0
  Journeys:      1
  Agents:        ~100
  Journey elems: 2
  Route:         1 distribution, 0 checkpoint, 1 exit
  Sequence:      jps-distributions_0 -> jps-exits_0
    jps-distributions_0: 100 agents
Using fallback logic: No journeys defined
Processing with parameters: {'number': 100, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'radius_distribution': 'constant', 'v0_distribution': 'gaussian', 'use_flow_spawning': False, 'v0_std': 0.2}
Using default parameters: v0=1.2, radius=0.15, n_agents=100

Distribution jps-distributions_0: {'number': 100, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'percentage': None, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_premovement': False, 'premovement_distributio

## Density at the room exit vs in the corridor

In [5]:
room_exit_box = ((3.0, 5.0), (5.0, 4.5))  # x in 3..5, y in 4.5..5
corridor_box = ((3.0, 17.0), (5.0, 15.0))  # x in 3..5, y in 15..17

def points_in(df, x_range, y_range):
    return df[(df.x >= x_range[0]) & (df.x <= x_range[1]) &
              (df.y >= y_range[0]) & (df.y <= y_range[1])]

AREA_ROOM_EXIT = 2.0 * 0.5
AREA_CORRIDOR = 2.0 * 2.0
rows = []
for frame, sub in df.groupby('frame'):
    rows.append({
        'time_s': frame / result.frame_rate,
        'rho_room_exit': len(points_in(sub, (3.0, 5.0), (4.5, 5.0))) / AREA_ROOM_EXIT,
        'rho_corridor': len(points_in(sub, (3.0, 5.0), (15.0, 17.0))) / AREA_CORRIDOR,
    })
density = pd.DataFrame(rows)
density.head()

,time_s,rho_room_exit,rho_corridor
0,0.0,0.0,0.0
1,0.1,1.0,0.0
2,0.2,2.0,0.0
3,0.3,3.0,0.0
4,0.4,4.0,0.0


## Plot densities over time

In [6]:
fig, ax = plt.subplots()
ax.plot(density.time_s, density.rho_room_exit, label='room exit (3..5, 4.5..5)')
ax.plot(density.time_s, density.rho_corridor, label='corridor (3..5, 15..17)')
ax.set_xlabel('time [s]'); ax.set_ylabel('density [1/m2]')
ax.legend()
plt.show()

## Acceptance

In [7]:
peak_room = density.rho_room_exit.max()
peak_corridor = density.rho_corridor.max()
print(f'peak room-exit density = {peak_room:.2f} /m2')
print(f'peak corridor density = {peak_corridor:.2f} /m2')
assert peak_room > peak_corridor, (peak_room, peak_corridor)
result.cleanup()

peak room-exit density = 7.00 /m2
peak corridor density = 2.75 /m2
